In [ ]:
# Local imports for notebook execution
import json
import os
import pickle
import re
import sys
from pathlib import Path
from typing import Dict, List, Optional

NOTEBOOK_DIR = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    direct = candidate / 'wvs_notebook_helpers.py'
    nested = candidate / 'paper_reproduction' / 'worldvalue_quantile' / 'wvs_notebook_helpers.py'
    if direct.exists():
        NOTEBOOK_DIR = candidate
        break
    if nested.exists():
        NOTEBOOK_DIR = nested.parent
        break
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate paper_reproduction/worldvalue_quantile.')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np
import pandas as pd

from wvs_notebook_helpers import (
    ensure_worldvalue_inputs,
    filter_mapping_to_questions,
    find_repo_root,
    install_numpy_pickle_compat,
    load_retained_questions,
    worldvalue_restore_instructions,
)

import wvs_data_preparation as dp

install_numpy_pickle_compat()


## Reproduction Notes
This notebook builds the cleaned WorldValue human-response artifacts, prompt inputs, and uniform baseline used by the quantile analysis.

- The canonical paper subset is `data/worldvalue/retained_questions_235.json`.
- The committed `choices_to_numeric.json` is the mapping consumed downstream; the local regeneration helpers remain here for reproducibility.
- Required raw inputs come from `datasets/worldvalue_data.zip`. Restore them at the repository root with `unzip -o datasets/worldvalue_data.zip`.
- The legacy failing uniform-construction branch has been retired from the main execution path.


## Persona Dataset Creation

### Repository Root And Dataset Setup
The notebook locates the repository root automatically, verifies that the extracted WorldValue inputs are present, and prints the exact restore archive if anything is missing.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = find_repo_root(Path.cwd())
CLEAN_OUTPUTS_DIR = ROOT / 'data' / 'worldvalue' / 'synthetic answers' / 'clean'
CLEAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

required_paths = ensure_worldvalue_inputs(ROOT)
print('Using repository root:', ROOT)
print(worldvalue_restore_instructions(ROOT))
for key in ['retained_questions', 'choices_to_numeric', 'question_metadata', 'codebook', 'answer_adjustment', 'wvs_raw_zip']:
    print(f'{key}:', required_paths[key])


In [ ]:
questions = load_retained_questions()
print(f'Retained question ids loaded: {len(questions)}')
questions[:10]


In [ ]:

# ---------- Helpers: code -> text mappers (WVS-7) ----------
SEX_Q260 = {1: "male", 2: "female", "Male": "male", "Female": "female"}

URBAN_RURAL = {1: "urban", 2: "rural", "Urban": "urban", "Rural": "rural"} # This is observator-reported

MARITAL_Q273 = {
    1: "married",
    2: "living together as married",
    3: "divorced",
    4: "separated",
    5: "widowed",
    6: "single",
}

# ISCED 2011 levels used in WVS-7 (Q275)
EDU_Q275 = {
    0: "no formal education / early childhood",
    1: "primary education",
    2: "lower secondary education",
    3: "upper secondary education",
    4: "post-secondary non-tertiary education",
    5: "short-cycle tertiary education",
    6: "bachelor’s degree or equivalent",
    7: "master’s degree or equivalent",
    8: "doctoral degree or equivalent",
}

# minimal default map (edit/extend with your country-specific labels if you have them)
RELIGION_Q289 = {
    0: "no religion",
    1: "Roman Catholic",
    2: "Protestant",
    3: "Orthodox",
    4: "Jewish",
    5: "Muslim",
    6: "Hindu",
    7: "Buddhist",
    8: "other religion"
}

# --- Sector map (edit if you have your own labels) ---
SECTOR_Q284 = {
    1: "the government/public sector",
    2: "the private business/industry sector",
    3: "the private non-profit sector",
}

# Minimal cleaner so we don't print junk like "Not asked"
EMPTY_TOKENS = {"", "na", "n/a", "nan", "none", "not asked", "not applicable",
                "dk", "refused", "dont know", "don't know"}
def clean_token(x):
    if x is None: return None
    s = str(x).strip(" ;,.'\"").strip()
    return None if s.lower() in EMPTY_TOKENS else s

def sector_text(val):
    iv = row_to_int(val)
    if iv in SECTOR_Q284:
        return SECTOR_Q284[iv]
    return clean_token(val)

def ideology_from_q240(val):
    """Q240: Left-Right self placement (1..10)."""
    try:
        x = float(val)
    except (TypeError, ValueError):
        return None
    if x <= 3:  return "left-leaning"
    if x >= 8:  return "right-leaning"
    return "centrist"

def norm(v):
    if v is None: return None
    if isinstance(v, float) and pd.isna(v): return None
    s = str(v).strip()
    return s if s else None

def age_band_from(row):
    """
    Prefer X003R (text like '45-54 years'); else use Q262 numeric age to map to bands.
    """
    x003r = norm(row.get("X003R"))
    if x003r:
        return x003r.replace(" years", "")
    try:
        age = int(row.get("Q262"))
    except (TypeError, ValueError):
        return None
    bins = [(18,24),(25,34),(35,44),(45,54),(55,64),(65,120)]
    for lo, hi in bins:
        if lo <= age <= hi:
            return f"{lo}–{hi}"
    return None

def immigrant_phrase(q263):
    """
    Q263 text is often already human-readable; handle common cases.
    """
    s = norm(q263)
    if not s: return None
    low = s.lower()
    if "born in this country" in low:
        return "you were born in this country (not an immigrant)"
    if "immigrant" in low or "born outside" in low:
        return "you are an immigrant to this country (born outside)"
    return s

def edu_phrase(q275):
    v = row_to_int(q275)
    if v is not None and v in EDU_Q275:
        return EDU_Q275[v]
    txt = norm(q275)
    return txt

def row_to_int(x):
    if isinstance(x, (int,)) and not pd.isna(x):
        return int(x)
    try:
        if isinstance(x, float) and pd.isna(x): return None
        return int(float(x))
    except Exception:
        return None

def pick_first_existing(row, cols, default=None):
    for c in cols:
        if c in row and norm(row[c]) is not None:
            return row[c]
    return default

def build_wvs_prompt(row,
                     question_text=None,
                     options_text=None,
                     force_country=None):
    parts = []

    # Geography
    country = force_country or norm(row.get("B_COUNTRY"))
    parts.append(f"Pretend that you reside in {country}" if country else "Pretend that you reside in this country")

    # Urbanicity & settlement
    urb_val = row.get("H_URBRURAL")
    urb = URBAN_RURAL.get(urb_val, norm(urb_val))
    settlement = norm(row.get("H_SETTLEMENT"))
    tsize = norm(row.get("G_TOWNSIZE"))
    geo_bits = []
    if settlement: geo_bits.append(settlement.lower())
    if urb: geo_bits.append(urb)
    if tsize: geo_bits.append(f"with a town size of {tsize}")
    if geo_bits:
        parts.append(f"You live in a {' '.join(geo_bits)} area")

    # Sex & age
    sex = SEX_Q260.get(row.get("Q260"), norm(row.get("Q260")))
    age_band = age_band_from(row)
    if sex and age_band:
        parts.append(f"You are {sex}, your age is between {age_band}")
    elif sex:
        parts.append(f"You are {sex}")
    elif age_band:
        parts.append(f"Your age is between {age_band}")

    # Marital
    marital = MARITAL_Q273.get(row.get("Q273"), norm(row.get("Q273")))
    if marital:
        parts.append(f"and you are {marital}")

    # Language
    lang = norm(row.get("S_INTLANGUAGE"))
    if lang:
        parts.append(f"You normally speak {lang} at home")

    # Migration background
    imm = immigrant_phrase(row.get("Q263"))
    if imm:
        parts.append(f"In terms of migration background, {imm}")

    # Education
    edu_code = row.get("Q275")
    edu_txt = EDU_Q275.get(row_to_int(edu_code), norm(edu_code))
    if edu_txt:
        parts.append(f"In terms of education, you attained {edu_txt}")

    # Employment status (keep)
    emp = pick_first_existing(row, ["Q279", "Q280"])
    emp_txt = clean_token(emp)
    if emp_txt:
        parts.append(f"Your current employment status is: {emp_txt}")

    # ✅ Sector only (Q284) — include when present, skip otherwise
    sec_txt = sector_text(row.get("Q284"))
    if sec_txt:
        parts.append(f"You work in {sec_txt}")

    # Religion attendance (Q171)
    attend_map = {
        1:"more than once a week", 2:"once a week", 3:"once a month",
        4:"only on special holy days", 5:"once a year", 6:"less often",
        7:"never, practically never"
    }
    attend = attend_map.get(row_to_int(row.get("Q171")), norm(row.get("Q171")))
    if attend:
        parts.append(f"You attend religious services {attend}")

    # Religion / denomination (Q289 or Q289CS9)
    rel_code = row.get("Q289")
    rel_txt_cs9 = norm(row.get("Q289CS9"))
    rel_txt = rel_txt_cs9 or RELIGION_Q289.get(row_to_int(rel_code), norm(rel_code))
    if rel_txt:
        parts.append(f"You belong to {rel_txt}")

    # Ideology (Q240)
    ideol = ideology_from_q240(row.get("Q240"))
    if ideol:
        parts.append(f"Politically, you are {ideol} on a 1–10 left–right scale")

    # Final stitch
    preamble = ". ".join(p for p in parts if p).strip()
    if preamble and not preamble.endswith("."):
        preamble += "."
    return preamble


In [ ]:
raw_dataset_path = 'data/worldvaluesbench/F00011356-WVS_Cross-National_Wave_7_csv_v6_0.zip'
df = pd.read_csv(raw_dataset_path, low_memory=False)


In [ ]:
df = df.loc[df['D_INTERVIEW'] != 858069901]
# Codebook contains the mapping from integer answer choice to the actual natural language answer
codebook = {}
with open(f'data/worldvaluesbench/dataset_construction/codebook.json', 'r') as f:
    codebook = json.load(f)

# Load question metadata
question_metadata = {}
with open(f'data/worldvaluesbench/dataset_construction/question_metadata.json', 'r') as f:
    question_metadata = json.load(f)

# data transformation
# filter required questions
req_question_keys = list(question_metadata.keys())
df = df[req_question_keys]

    # get preprocessing mappings
preprocess_map = {}
with open(f'data/worldvaluesbench/dataset_construction/answer_adjustment.json', 'r') as f:
    preprocess_map = json.load(f)


In [ ]:
# Preprocess all the answers
for question_key in req_question_keys:
    df[question_key] = df[question_key].map(lambda val: dp.preprocess(val, question_key, preprocess_map))

# Process all the answers
for question_key in req_question_keys:
    df[question_key] = df[question_key].map(lambda val: dp.process(val, question_key, question_metadata, codebook))

 # Persona columns are the columns which contain the demographic information of the user
demographic_columns = [q for q in req_question_keys if question_metadata[q]['use_case'] == 'demographic']
demographic_columns.remove('D_INTERVIEW')
demographic_columns = ['D_INTERVIEW'] + demographic_columns
persona_df = df[demographic_columns]
persona_df.set_index('D_INTERVIEW')

persona_df["PROMPT"] = persona_df.apply(lambda r: build_wvs_prompt(r), axis=1)


## Prompt Dataset Creation

In [ ]:
# ---- Vectorized augmenter over a slice (no row-wise apply) ----
def augment_prompt_vec(df_slice: pd.DataFrame, qmeta: dict, qid: str,
                       require_double_brackets: bool = True,
                       base_col: str = "PROMPT") -> pd.Series:
    base = df_slice[base_col].astype("string", copy=False).fillna("")
    m = (qmeta or {}).get(qid, {}) or {}
    qtext = str(m.get("question") or f"[{qid}] Question text not provided.").strip()
    amin, amax = m.get("answer_scale_min"), m.get("answer_scale_max")
    try: amin = int(amin) if amin not in (None, "") else None
    except: amin = None
    try: amax = int(amax) if amax not in (None, "") else None
    except: amax = None
    if amin is not None and amax is not None:
        instr = (f"Please respond with a single number from {amin} to {amax} "
                 f"{'in double square brackets, e.g., [[%d]].' % amin if require_double_brackets else 'with no extra text.'}")
    else:
        instr = "Please respond with a single number."
    add = f"{qtext}\n\n{instr}"
    sep = "\n\n"
    has_base = base.str.len() > 0
    return base.where(~has_base, base + sep + add).where(has_base, add).astype("string", copy=False)

# ---------- Config: which demographic columns to keep (inputs to your persona prompt) ----------
DEMOG_CANDIDATES = [
    # identifiers / tech
    "D_INTERVIEW","A_YEAR","B_COUNTRY","J_INTDATE","N_TOWN",
    # geography / settlement
    "G_TOWNSIZE","H_SETTLEMENT","H_URBRURAL",
    # language / literacy
    "S_INTLANGUAGE","E1_LITERACY",
    # key sociodemographics used in prompt builder
    "Q260",          # sex
    "Q261","Q262",   # age / age bands if present
    "Q263",          # migration background
    "Q273",          # marital
    "Q275",          # education
    "Q279","Q280",   # employment status (two variants, keep whichever exists)
    "Q284",          # sector
    "Q171",          # religion attendance
    "Q289","Q289CS9",# religion / denomination (coded + text)
    # if you keep any other columns in your persona prompt, add them here
    "PROMPT"         # base persona prompt (if missing, we’ll create empty string)
]

def build_sampled_synth_profiles_dict(
    persona_df: pd.DataFrame,
    qmeta: dict,
    sample_n: int = 500,
    random_seed: int = 123,
    id_col: str = "D_INTERVIEW",
    only_q_pattern: str = r"Q(\d+)",
    q_lo: int = 1,
    q_hi: int = 259,
    require_double_brackets: bool = True,
    keep_cols: Optional[List[str]] = None,
    same_sample_all_qids: bool = False
) -> Dict[str, pd.DataFrame]:
    """
    Returns a dict {qid: DataFrame} where each DataFrame has:
        [id_col] + selected demographic columns + PROMPT(augmented for that qid)
    For each qid, it randomly samples `sample_n` rows (without replacement).
    If same_sample_all_qids=True, the same 500 respondents are used for every qid.
    """

    # ensure base PROMPT exists (empty OK)
    if "PROMPT" not in persona_df.columns:
        persona_df = persona_df.copy()
        persona_df["PROMPT"] = ""

    # derive keep_cols: ID + available demographic columns + PROMPT
    if keep_cols is None:
        # keep only those that actually exist
        keep_cols = [c for c in DEMOG_CANDIDATES if c in persona_df.columns]
        # ensure id_col is first and included
        if id_col not in keep_cols:
            keep_cols = [id_col] + keep_cols
        # move PROMPT to the end for readability
        if "PROMPT" in keep_cols:
            keep_cols = [c for c in keep_cols if c != "PROMPT"] + ["PROMPT"]

    # pick QIDs
    qids = []
    for k in qmeta.keys():
        m = re.fullmatch(only_q_pattern, str(k))
        if not m:
            continue
        i = int(m.group(1))
        if q_lo <= i <= q_hi:
            qids.append(f"Q{i}")
    qids.sort(key=lambda s: int(s[1:]))

    n = len(persona_df)
    if sample_n > n:
        raise ValueError(f"sample_n ({sample_n}) > dataset size ({n}). Reduce sample_n.")

    rng = np.random.default_rng(random_seed)
    # precompute one sample if we want the same 500 across all QIDs
    shared_idx = None
    if same_sample_all_qids:
        shared_idx = rng.choice(persona_df.index.to_numpy(), size=sample_n, replace=False)

    out: Dict[str, pd.DataFrame] = {}
    for j, qid in enumerate(qids):
        if same_sample_all_qids:
            idx = shared_idx
        else:
            # different random sample per question, using stable but varied seeds
            # shift the bit generator to avoid correlated draws
            rng_local = np.random.default_rng(random_seed + j * 9973)
            idx = rng_local.choice(persona_df.index.to_numpy(), size=sample_n, replace=False)

        # slice demographics
        df_slice = persona_df.loc[idx, keep_cols].copy()

        # augment PROMPT for this qid (vectorized over the slice)
        df_slice["PROMPT"] = augment_prompt_vec(df_slice, qmeta, qid, require_double_brackets=require_double_brackets)

        out[qid] = df_slice

    return out

def save_qid_dict_pickle(qid_to_df: Dict[str, pd.DataFrame], out_path: str, overwrite: bool = True) -> str:
    """
    Save the dictionary {qid: DataFrame} to a single pickle file.
    """
    p = Path(out_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists() and not overwrite:
        raise FileExistsError(f"Refusing to overwrite: {p}")
    with open(p, "wb") as f:
        pickle.dump(qid_to_df, f, protocol=pickle.HIGHEST_PROTOCOL)
    return str(p)

# --------------------------- USAGE ---------------------------
# persona_df : your full (~90k) persona DataFrame
# question_metadata : your metadata dict with "question", "answer_scale_min/max", etc.

# Build the sampled dictionary (500 rows per QID, independent samples per QID)
# If you prefer the same 500 respondents for every QID, set same_sample_all_qids=True.
qid_to_df = build_sampled_synth_profiles_dict(
    persona_df=persona_df,
    qmeta=question_metadata,
    sample_n=500,
    random_seed=123,
    id_col="D_INTERVIEW",
    same_sample_all_qids=False   # change to True if you want a single shared sample
)

# Save it as one small pickle (dictionary)
out_path = save_qid_dict_pickle(qid_to_df, "data/worldvalue/synthetic_profiles.pkl", overwrite=True)
print("Wrote:", out_path, "| QIDs in dict:", len(qid_to_df))
# Later:
# import pickle
# with open(out_path, "rb") as f:
#     sampled = pickle.load(f)  # dict: qid -> DataFrame (demographics + PROMPT)
# list(sampled)[:5], sampled["Q114"].head()


In [ ]:
qid_to_df['Q163']['PROMPT'].iloc[0]

## Conversion Map Creation

In [ ]:
import re
import numpy as np
import pandas as pd

# ------------------------- tiny helpers -------------------------
def _norm(s):
    return re.sub(r"\s+", " ", str(s or "")).strip().lower()

def _extract_endpoints(qtext: str):
    """
    Try to detect endpoint labels like:
    "1 meaning 'Very good' and 5 meaning 'Very bad'"
    Returns (left_label, right_label) or (None, None).
    """
    txt = _norm(qtext)
    m = re.search(r"1\s+meaning\s+'([^']+)'[^0-9]*?(\d{1,2})\s+meaning\s+'([^']+)'", txt)
    if m:
        return m.group(1), m.group(3)
    m = re.search(r"1\s*(?:=|stands for)\s*'([^']+)'[^0-9]*?(\d{1,2})\s*(?:=|stands for)\s*'([^']+)'", txt)
    if m:
        return m.group(1), m.group(3)
    # crude fallback: first and last quoted phrases
    quotes = re.findall(r"'([^']+)'", txt)
    if len(quotes) >= 2:
        return quotes[0], quotes[-1]
    return None, None

# lightweight lexicons
_POS = {
    "good","very good","benefit","benefits","well","very well","trust","support",
    "agree","strongly agree","important","very important","safe","happiness",
    "satisfied","positive","opportunity","prosperity","well-being","tolerance"
}
_NEG = {
    "bad","very bad","risk","very high risk","corruption","bribe","crime","violence",
    "pollution","inequality","distrust","threat","danger","nothing","none","never",
    "not at all","very little","not too much","worse","decline"
}
# question-topic cues
_MORE_POS = {"benefit", "trust", "happiness", "safety", "well-being", "freedom", "prosperity", "respect", "education", "health"}
_MORE_NEG = {"corruption", "bribe", "crime", "violence", "risk", "pollution", "inequality", "unemployment", "poverty", "threat", "danger", "distrust"}

def _valence(phrase: str) -> int:
    """Very small heuristic polarity score for endpoint labels."""
    p = _norm(phrase)
    sc = 0
    for w in _POS:
        if w in p: sc += 2
    for w in _NEG:
        if w in p: sc -= 2
    if "not " in p or "no " in p:
        sc -= 1
    return sc

def _subject_orientation(qtext: str) -> int:
    """
    +1 if 'more' likely positive, -1 if 'more' likely negative, 0 unknown.
    Uses question stem cues only.
    """
    t = _norm(qtext)
    pos_hit = any(w in t for w in _MORE_POS)
    neg_hit = any(w in t for w in _MORE_NEG)
    if pos_hit and not neg_hit: return +1
    if neg_hit and not pos_hit: return -1
    return 0

def _decide_positive_end(qtext: str, amin: int, amax: int) -> int:
    """
    Return +1 if lower code (amin) should map to +1, return -1 if upper code (amax) should map to +1.
    Strategy (simple):
      1) If endpoints text found: pick the more positive label.
      2) Else, use subject orientation.
      3) Else, default to WVS-like convention: lower code is more favorable.
    """
    L, R = _extract_endpoints(qtext)
    if L or R:
        vl, vr = _valence(L or ""), _valence(R or "")
        if vl > vr: return +1     # left (low code) is more positive
        if vr > vl: return -1     # right (high code) is more positive
    orient = _subject_orientation(qtext)
    if orient == +1: return -1    # more = positive → high code is positive
    if orient == -1: return +1    # more = negative → low code is positive
    return +1                      # fallback

# ------------------- build & apply conversion -------------------
def build_conversion_map(qmeta: dict, q_lo: int = 1, q_hi: int = 259,
                         manual_flip: dict | None = None) -> dict:
    """
    Build { 'Qk': {code: scaled_value_in_[-1,1]} } for Q_lo..Q_hi using simple text cues.
    `manual_flip`: optional dict like {'Q120': True, ...} to force flipping (high code = +1).
    """
    conv = {}
    manual_flip = manual_flip or {}
    # collect target qids
    qids = []
    for k in qmeta.keys():
        m = re.fullmatch(r"Q(\d+)", str(k))
        if not m:
            continue
        i = int(m.group(1))
        if q_lo <= i <= q_hi:
            qids.append(f"Q{i}")
    qids.sort(key=lambda s: int(s[1:]))

    for qid in qids:
        meta = qmeta.get(qid, {}) or {}
        qtext = meta.get("question") or ""
        amin = meta.get("answer_scale_min")
        amax = meta.get("answer_scale_max")
        try: amin = int(amin) if amin not in (None, "") else None
        except: amin = None
        try: amax = int(amax) if amax not in (None, "") else None
        except: amax = None
        if amin is None or amax is None or amin == amax:
            continue

        # which end is positive?
        pos_end = _decide_positive_end(qtext, amin, amax)  # +1 -> low is +1; -1 -> high is +1
        if manual_flip.get(qid, False):
            pos_end = -pos_end

        step = (amax - amin)
        mp = {}
        for k in range(amin, amax + 1):
            t = (k - amin) / step  # 0..1 from low to high
            if pos_end == +1:
                val = 1.0 - 2.0 * t      # low→+1, high→-1
            else:
                val = -1.0 + 2.0 * t     # low→-1, high→+1
            mp[k] = float(np.round(val, 12))
        conv[qid] = mp
    return conv

def apply_conversion(responses_df: pd.DataFrame, conv: dict,
                     cols: list[str] | None = None,
                     missing_to=np.nan) -> pd.DataFrame:
    """
    Add Q*_SCALED columns to responses_df using 'conv'.
    Any value not present in conv[q] mapping (DK/Refused/out-of-range) -> missing_to (default NaN).
    """
    df = responses_df.copy()
    if cols is None:
        cols = [c for c in df.columns if re.fullmatch(r"Q(\d+)", str(c)) and 1 <= int(c[1:]) <= 259]
    for q in cols:
        mp = conv.get(q)
        if not mp:
            continue
        ser = pd.to_numeric(df[q], errors="coerce")
        out = np.full(len(df), missing_to, dtype=float)
        mask = ser.notna() & ser.astype(int).isin(mp.keys())
        idx = mask[mask].index
        out[idx] = [mp[int(v)] for v in ser.loc[idx].astype(int).values]
        df[q + "_SCALED"] = out
    return df

# ------------------------- usage -------------------------
# 1) Build conversion map from your metadata dict `qmeta` (Q1..Q259)
conv = build_conversion_map(question_metadata)

def save_conversion_map_json(conv: dict, out_path: str, overwrite: bool = False, pretty: bool = True) -> str:
    """
    Save the Q1–Q259 conversion map to JSON.
    JSON requires string keys, so inner numeric codes are stringified.
    conv: {'Q1': {1: 1.0, 2: 0.3333, ...}, 'Q2': {...}, ...}
    """
    p = Path(out_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists() and not overwrite:
        raise FileExistsError(f"Refusing to overwrite existing file: {p}")

    conv_jsonable = {qid: {str(k): float(v) for k, v in mp.items()} for qid, mp in conv.items()}
    with open(p, "w", encoding="utf-8") as f:
        if pretty:
            json.dump(conv_jsonable, f, ensure_ascii=False, indent=2, sort_keys=True)
        else:
            json.dump(conv_jsonable, f, ensure_ascii=False, separators=(",", ":"))
    return str(p)

def load_conversion_map_json(in_path: str, as_int_keys: bool = True) -> dict:
    """
    Load the conversion map from JSON. If as_int_keys=True,
    convert inner keys back to int.
    """
    p = Path(in_path)
    with open(p, "r", encoding="utf-8") as f:
        data = json.load(f)
    if as_int_keys:
        return {qid: {int(k): float(v) for k, v in mp.items()} for qid, mp in data.items()}
    return data

# ===== Example =====
# conv = build_conversion_map(qmeta)  # from earlier
path = save_conversion_map_json(conv, "data/worldvalue/choices_to_numeric.json", overwrite=True)
# conv_loaded = load_conversion_map_json(path)  # inner keys restored to int


In [ ]:
with open('data/worldvalue/synthetic answers/raw/synthetic_answers_raw_llama-3.3-1.pkl', "rb") as f:
     sampled = pickle.load(f)

### Population Data Cleaning
- This used the synthetic profile that is actually later created.

In [ ]:
# load questions and profiles
questions = load_retained_questions()
with open('data/worldvalue/synthetic_profiles.pkl', 'rb') as f:
    synthetic_profiles = pickle.load(f) # key: question id, value: synthetic profiles
with open('data/worldvalue/choices_to_numeric.json', 'r') as f:
    choices_to_numeric = json.load(f) # key: question id, value: choices to numeric mapping

In [ ]:
persona_df

In [ ]:
# Filter the DataFrame to keep only the specified columns
filtered_df = df[['D_INTERVIEW'] + questions]

# Display the first few rows of the filtered_df to verify the filtering.
display(filtered_df.head())

In [ ]:
_WVS_MISSING = {-1, -2, -3, -5}

def population_numeric_means_from_wide(
    df_wide: pd.DataFrame,
    choices_to_numeric: dict,      # {'Q1': {'1': -1.0, '2': 0.0, '3': 1.0}, ...}
    drop_missing_codes: bool = True,
    out_clean_dict_path: str | None = None,
    out_clean_sum_path: str | None = None, # optional: save per-Qid numeric Series dict
):
    """
    Convert a wide WVS-style DataFrame to numeric answers and return per-question means.

    Returns
    -------
    summary : pd.DataFrame
        index = question ids; columns = ['n_valid','mean','std'].
    cleaned : dict[str, pd.Series]
        Only returned if out_clean_dict_path is not None:
        {qid: pd.Series(numeric_scores, index=D_INTERVIEW)}
    """
    # normalize mapping keys -> str; values -> float
    c2n = {q: {str(k): float(v) for k, v in mp.items()} for q, mp in choices_to_numeric.items()}

    # choose question columns that have a mapping and exist in df
    qids = [q for q in c2n.keys() if q in df_wide.columns]
    if not qids:
        raise ValueError("No question columns in df_wide match choices_to_numeric keys.")

    cleaned = {}  # optional payload
    rows = []
    for qid in qids:
        s_raw = df_wide[qid]

        # parse numeric codes first
        s_num = pd.to_numeric(s_raw, errors="coerce")  # floats; NaN on bad
        mask = s_num.notna()
        if drop_missing_codes:
            mask &= ~s_num.isin(_WVS_MISSING)

        # filter to valid rows
        s_num = s_num[mask]

        # cast to str codes AFTER filtering (safe) and map to [-1,1]
        s_codes = s_num.astype(np.int64).astype(str)
        mp = c2n[qid]
        s_val = s_codes.map(mp).astype(float)

        # summary stats (skip NaNs if a code was missing from the map)
        n_valid = int(s_val.notna().sum())
        mean = float(s_val.mean(skipna=True))
        std  = float(s_val.std(skipna=True, ddof=1)) if n_valid > 1 else np.nan
        rows.append((qid, n_valid, mean, std))

        if out_clean_dict_path is not None:
            # keep respondent ids if present; else default RangeIndex
            if "D_INTERVIEW" in df_wide.columns:
                idx = df_wide.loc[mask, "D_INTERVIEW"].values
            else:
                idx = s_val.index.values
            cleaned[qid] = pd.Series(s_val.values, index=idx, name=qid)

    # build summary dataframe
    summary = pd.DataFrame(rows, columns=["qid", "n_valid", "mean", "std"]).set_index("qid")

    # optional save of cleaned dict
    if out_clean_dict_path:
        os.makedirs(os.path.dirname(out_clean_dict_path) or ".", exist_ok=True)
        with open(out_clean_dict_path, "wb") as f:
            pickle.dump(cleaned, f)
        print(f"[OK] Saved cleaned per-question series → {out_clean_dict_path}")

    if out_clean_sum_path:
        os.makedirs(os.path.dirname(out_clean_sum_path) or ".", exist_ok=True)
        with open(out_clean_sum_path, "wb") as f:
            pickle.dump(summary, f)
        print(f"[OK] Saved cleaned per-question series → {out_clean_sum_path}")

    return summary if out_clean_dict_path is None else (summary, cleaned)


In [ ]:
# 1) Just get per-question means (and counts/std)
summary = population_numeric_means_from_wide(filtered_df, choices_to_numeric)
print(summary.head())
# -> DataFrame with index=QIDs, columns: n_valid, mean, std

# 2) Also save the cleaned numeric responses per question

summary, cleaned = population_numeric_means_from_wide(
    filtered_df,
    choices_to_numeric,
    out_clean_dict_path="data/worldvalue/full_population_response_clean.pkl",
    out_clean_sum_path="data/worldvalue/full_population_response_summary.pkl"
)
print(summary.loc[["Q1","Q2","Q90"]])
# cleaned["Q90"] is a Series of numeric values indexed by D_INTERVIEW (if present)


In [ ]:
# Create a dictionary to store the synthetic responses organized by question
synthetic_responses_by_question = {}

# Iterate through each question in synthetic_profiles
for question_key, profile_df in synthetic_profiles.items():
    # Get the D_INTERVIEW IDs for the current question's synthetic profiles
    interview_ids = profile_df['D_INTERVIEW'].tolist()

    # Filter the filtered_df to get the responses for these interview IDs and the current question
    # Ensure the question_key exists in filtered_df before selecting the column
    if question_key in filtered_df.columns:
        question_columns = ['D_INTERVIEW', question_key]
        responses_for_question = filtered_df[filtered_df['D_INTERVIEW'].isin(interview_ids)][question_columns]

        # Store the responses in the dictionary
        synthetic_responses_by_question[question_key] = responses_for_question
    else:
        print(f"Warning: Question {question_key} not found in filtered_df.")

In [ ]:
# Define the output path
output_dir = 'data/worldvalue'
output_filename = 'population_response_raw.pkl'
output_path = os.path.join(output_dir, output_filename)

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save the dictionary to a pickle file
with open(output_path, 'wb') as f:
    pickle.dump(synthetic_responses_by_question, f)

print(f"Organized responses saved to {output_path}")

In [ ]:
with open('data/worldvalue/population_response_raw.pkl', 'rb') as f:
    pop_response = pickle.load(f)

In [ ]:
import os, pickle, numpy as np, pandas as pd

_WVS_MISSING = {-1, -2, -3, -5}

def process_population_numeric_series(
    pop_response,                 # {'Q1': DataFrame(['D_INTERVIEW','Q1']), ...}
    choices_to_numeric,           # {'Q1': {'1':-1.0,'2':0.0,'3':1.0}, ...}
    out_path="population_response_clean.pkl",
    drop_missing_codes=True
):
    # resume if exists
    numeric_out = {}
    if os.path.exists(out_path):
        try:
            with open(out_path, "rb") as f:
                numeric_out = pickle.load(f) or {}
        except Exception:
            numeric_out = {}

    # normalize mapping keys to str once
    c2n = {q: {str(k): float(v) for k, v in mp.items()} for q, mp in choices_to_numeric.items()}

    for qid, df in pop_response.items():
        if qid in numeric_out:
            continue
        if not isinstance(df, pd.DataFrame) or not {"D_INTERVIEW", qid}.issubset(df.columns):
            print(f"[skip] {qid}: expected DataFrame with columns ['D_INTERVIEW','{qid}'].")
            continue

        # 1) numeric parse first, then mask using the parsed series
        s_raw = df[qid]
        idx = df["D_INTERVIEW"]

        s_num = pd.to_numeric(s_raw, errors="coerce")          # -> floats with NaN on bad rows
        mask = s_num.notna()                                   # drop NaNs from parsing
        if drop_missing_codes:
            mask &= ~s_num.isin(_WVS_MISSING)                  # drop WVS non-substantive

        # apply mask
        s_num = s_num[mask]
        idx_valid = idx[mask]

        # 2) cast to int only after NA-free filtering
        #    (safe because WVS codes are integers 1..10)
        s_codes = s_num.astype(np.int64).astype(str)

        # 3) map codes -> [-1,1]
        mp = c2n.get(qid, {})
        if not mp:
            print(f"[warn] {qid}: no mapping in choices_to_numeric; values will be NaN.")
        vals = s_codes.map(mp).astype(float)

        # 4) store as a Series indexed by respondent id
        series = pd.Series(vals.values, index=idx_valid.values, name=qid)
        numeric_out[qid] = series

        # checkpoint
        with open(out_path, "wb") as f:
            pickle.dump(numeric_out, f)
        print(f"[ok] {qid}: kept {series.size} rows, mean={np.nanmean(series.values):.3f}, std={np.nanstd(series.values):.3f}")

    print(f"Population conversion completed → {out_path}")
    return numeric_out


In [ ]:
OUT_PATH = "data/worldvalue/population_response_clean.pkl"

numeric_pop = process_population_numeric_series(
    pop_response=pop_response,
    choices_to_numeric=choices_to_numeric,
    out_path=OUT_PATH,
    drop_missing_codes=True
)

# Quick peek
print("Converted questions:", len(numeric_pop))
for q in list(numeric_pop.keys())[:5]:
    s = numeric_pop[q]
    print(f"{q}: n={s.size}, mean={np.nanmean(s):.3f}, std={np.nanstd(s):.3f}")


## Uniform Simulator Dataset Construction

In [ ]:
# ---------- small helpers ----------
def _safe_int(x, default=None):
    if x is None: return default
    s = str(x).strip()
    if s == "": return default
    try:
        return int(float(s))
    except Exception:
        return default

def filter_keys_in_order(d: dict, keys_in_order):
    """Return a new dict with only keys in keys_in_order, preserving order."""
    ks = [k for k in keys_in_order if k in d]
    return {k: d[k] for k in ks}

# ---------- build integer-code support from metadata or fallback to the map ----------
def code_support_from_metadata_or_map(question_metadata, choices_to_numeric, questions_subset=None):
    """
    Returns {qid: [code_str,...]} only for qids in questions_subset (if provided).
    """
    if questions_subset is None:
        qids_iter = list(question_metadata.keys())
    else:
        qids_iter = list(questions_subset)  # preserve given order

    support = {}
    for qid in qids_iter:
        meta = question_metadata.get(qid, {})
        a_min = _safe_int(meta.get("answer_scale_min", None), default=None)
        a_max = _safe_int(meta.get("answer_scale_max", None), default=None)

        codes = None
        if a_min is not None and a_max is not None:
            if a_max < a_min:
                a_min, a_max = a_max, a_min
            codes = [str(x) for x in range(a_min, a_max + 1)]

        if not codes:
            # fallback: use integer-like keys from the conversion map
            mp = choices_to_numeric.get(qid, {}) or {}
            ints = []
            for k in mp.keys():
                ki = _safe_int(k, default=None)
                if ki is not None:
                    ints.append(ki)
            if ints:
                ints = sorted(set(ints))
                codes = [str(x) for x in ints]

        if not codes:
            # final safe default (will still be validated by strict flag later)
            codes = ["1", "2"]

        support[qid] = codes
    return support

# ---------- uniform sampler over integer codes, then convert via your map ----------
def generate_uniform_from_codes(
    code_support, choices_to_numeric, n_per_q=500, seed=123, strict=True
):
    rng = np.random.default_rng(seed)
    out = {}
    for qid, codes in code_support.items():
        codes = list(codes) if codes else ["1", "2"]
        idx = rng.integers(0, len(codes), size=int(n_per_q))
        sampled_codes = [codes[i] for i in idx]

        mp = choices_to_numeric.get(qid, {}) or {}
        if strict:
            missing = sorted({c for c in set(sampled_codes) if c not in mp})
            if missing:
                raise ValueError(
                    f"[{qid}] Missing mapping for codes {missing}. "
                    f"Have mapping keys: {sorted(mp.keys())[:10]} ..."
                )
        vals = [float(mp.get(c, np.nan)) for c in sampled_codes]
        out[qid] = vals
    return out

def save_pickle(obj, path):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"[OK] Saved: {path}")


In [ ]:
# 1) Load the allowed question IDs (your screenshot)
questions = load_retained_questions()

# 2) Build code support ONLY for those questions
CODE_SUPPORT = code_support_from_metadata_or_map(
    question_metadata=question_metadata,
    choices_to_numeric=choices_to_numeric,
    questions_subset=questions
)

# (Optional) If you want to also filter your conversion map/metadata to this subset:
choices_to_numeric_sub = filter_keys_in_order(choices_to_numeric, questions)
# question_metadata_sub = filter_keys_in_order(question_metadata, questions)

# 3) Generate 500 profiles per question (uniform over integer codes), then convert
UNIFORM_BENCHMARK = generate_uniform_from_codes(
    code_support=CODE_SUPPORT,
    choices_to_numeric=choices_to_numeric_sub,
    n_per_q=500,
    seed=123,
    strict=True
)



In [ ]:

# Save
SAVE_PATH = str(CLEAN_OUTPUTS_DIR / "uniform_benchmark.pkl")
save_pickle(UNIFORM_BENCHMARK, SAVE_PATH)

# Quick sanity check: values must be from the mapped set for each qid
def _check(uniform_benchmark, choices_to_numeric, n_show=3):
    bad = []
    for qid, vals in list(uniform_benchmark.items())[:2000]:  # cap iteration if huge
        mp = choices_to_numeric.get(qid, {}) or {}
        allowed = {round(float(v), 10) for v in mp.values()}
        if not allowed:
            continue
        produced = [round(float(x), 10) for x in vals]
        if any(v not in allowed for v in produced):
            bad.append(qid)
    if bad:
        print("[WARN] Some questions produced values outside mapping:", bad[:10], "…")
    else:
        print("[OK] All sampled answers correspond to integer codes via the conversion map.")
    # peek one
    some_q = next(iter(uniform_benchmark.keys()))
    print(some_q, "first 8:", uniform_benchmark[some_q][:8])

_check(UNIFORM_BENCHMARK, choices_to_numeric)


In [ ]:
# Save the retained-question uniform benchmark if it has already been built above.
if 'UNIFORM_BENCHMARK' not in globals():
    print('UNIFORM_BENCHMARK is not defined yet. Run the retained-question uniform pipeline above if you want to regenerate it in this notebook run.')
else:
    SAVE_PATH = str(CLEAN_OUTPUTS_DIR / 'uniform_benchmark.pkl')
    save_pickle(UNIFORM_BENCHMARK, SAVE_PATH)
    some_q = next(iter(UNIFORM_BENCHMARK.keys()))
    print(some_q, 'first 8:', UNIFORM_BENCHMARK[some_q][:8])


In [ ]:
# Load question metadata
question_metadata = {}
with open(f'data/worldvaluesbench/dataset_construction/question_metadata.json', 'r') as f:
    question_metadata = json.load(f)
with open('data/worldvalue/choices_to_numeric.json', 'r') as f:
    choices_to_numeric = json.load(f) # key: question id, value: choices to numeric mapping

In [ ]:
# Legacy cell retired.
# The older `code_support_from_metadata(...)` path failed on empty metadata fields and is no longer part of the main reproduction flow.
print('Legacy uniform-construction cell retired. Use `code_support_from_metadata_or_map(...)` with retained questions instead.')
